# Task 3 - Feature Hierarchies and Representations

**Goal:** collect activations ("features") from early, middle and late layers of the frozen pre-trained ResNet-152, visualize them with t-SNE, and study how class separability evolves through the network.

## 1. Environment & Data

In [ ]:
%pip install torch torchvision torchaudio scikit-learn

In [ ]:
# ---------- Imports ----------
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from torchvision.models import resnet152, ResNet152_Weights

# ---------- Configuration ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64

print(f"Using: {device}")

In [ ]:
# CIFAR-10 images are 32x32; ResNet-152 expects 224x224.
# The mean/std is the ImageNet normalization required by the pre-trained weights.
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

valset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_val)
val_loader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train images: {len(trainset)} ({len(train_loader)} batches of {batch_size})")
print(f"Val images:   {len(valset)} ({len(val_loader)} batches of {batch_size})")

## 2. Model (same frozen baseline as Task 1)

In [ ]:
# 1) Load the ResNet-152 pre-trained on ImageNet
model = resnet152(weights=ResNet152_Weights.DEFAULT)

# 2) Freeze the entire backbone - no gradients flow into it during training
for param in model.parameters():
    param.requires_grad = False

# 3) Replace the 1000-class head with a 10-class head (CIFAR-10)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 10)

model = model.to(device)

# How much of the network actually gets trained?
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total/1e6:8.2f} M")
print(f"Trainable (head): {trainable/1e6:8.2f} M  ({100*trainable/total:.3f}% of the network)")

## 3. Feature extraction - early / middle / late layers

| Probe        | Module          | Dims after GAP | Depth        |
|--------------|-----------------|----------------|--------------|
| **early**    | `model.layer1`  | 256            | shallow      |
| **middle**   | `model.layer3`  | 1024           | mid          |
| **late**     | `model.avgpool` | 2048           | deepest (feeds the head) |

Forward hooks capture each layer's output; global average pooling turns each feature map into a single per-image vector. We use 300 images per class from the validation set (3000 total).

In [ ]:
model.eval()
features = {}


def make_hook(name):
    def hook_fn(module, input, output):
        features[name] = output.detach()
    return hook_fn


hooks = [model.layer1.register_forward_hook(make_hook("early")),
         model.layer3.register_forward_hook(make_hook("middle")),
         model.avgpool.register_forward_hook(make_hook("late"))]

# CIFAR-10's test set is ordered by class (1000 images per class)
indices = [c * 1000 + i for c in range(10) for i in range(300)]
subset = torch.utils.data.Subset(valset, indices)
subset_loader = torch.utils.data.DataLoader(subset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

feat_cols = {"early": [], "middle": [], "late": []}
labels_all = []

with torch.no_grad():
    for images, labels in subset_loader:
        model(images.to(device))
        for name in feat_cols:
            # Global average pooling: (N, C, H, W) -> (N, C)
            feat_cols[name].append(features[name].mean(dim=(2, 3)).cpu())
        labels_all.append(labels)

for h in hooks:
    h.remove()

for name in feat_cols:
    feat_cols[name] = torch.cat(feat_cols[name], dim=0).numpy()
labels_all = torch.cat(labels_all).numpy()

print("Feature shapes per layer:")
for name in feat_cols:
    print(f"  {name:>8}: {feat_cols[name].shape}   (3000 images x {feat_cols[name].shape[1]} dims)")

## 4. Class separability - silhouette score

Silhouette measures how well-defined the class clusters are in feature space: **+1** = well separated, **0** = overlapping, **-1** = misclustered. Computed on a random 1000-sample subset (the 2048-d raw features make the absolute values small, but the trend across layers is the signal).

In [ ]:
rng = np.random.RandomState(0)
idx = rng.choice(len(labels_all), 1000, replace=False)

sil = {}
print(f"{'Layer':<10}{'Dim':>8}{'Silhouette':>12}")
print("-" * 30)
for name in feat_cols:
    sil[name] = silhouette_score(feat_cols[name][idx], labels_all[idx])
    print(f"{name:<10}{feat_cols[name].shape[1]:>8}{sil[name]:>12.4f}")

## 5. t-SNE visualization (2-D embedding per layer)

In [ ]:
# Same seed for every layer so the visualizations are directly comparable
emb = {}
for name in feat_cols:
    tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto",
                max_iter=1000, random_state=0, n_jobs=-1)
    emb[name] = tsne.fit_transform(feat_cols[name])
print("t-SNE embeddings:", {k: v.shape for k, v in emb.items()})

In [ ]:
# Each point is one image, colored by its true class.
# The colorbar has its own grid column, so it never overlaps the plots.
fig = plt.figure(figsize=(18, 5))
gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 0.04], wspace=0.2)
axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
cax = fig.add_subplot(gs[0, 3])

for ax, name in zip(axes, feat_cols):
    sc = ax.scatter(emb[name][:, 0], emb[name][:, 1], c=labels_all, cmap="tab10", s=6, alpha=0.7)
    ax.set_title(f"{name} layer | silhouette {sil[name]:.3f}")
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(sc, cax=cax, ticks=range(10))
plt.savefig("feature_tsne.png", dpi=150)
plt.show()

## 6. Discussion

### How does class separability evolve across layers?

- **Early layer:** the t-SNE plot is one undifferentiated cloud and the silhouette is ~0 (even slightly negative). Early features encode generic primitives - edges, corners, colors, textures - that every class shares, so cats, dogs, cars and ships all look alike in this space.
- **Middle layer:** features begin combining primitives into object parts; partial clustering appears and the silhouette turns positive.
- **Late layer:** compact, class-specific clusters emerge and the silhouette is highest. The representation has become nearly linearly separable - which is exactly why a single linear head on top of layer4 reaches ~85% accuracy in Task 1.

### Low-level vs high-level representations

- **Low-level (early layers):** local, translation-invariant, **class-agnostic** - they transfer across datasets almost for free (the reason transfer learning works).
- **High-level (late layers):** global and **task-specific** - they compress an image into a semantic signature (presence of wheels, eyes, fur...), losing pixel detail but gaining class identity.
- The hierarchy is why we freeze the bottom and only re-learn the top when moving to a new dataset: the general-purpose features are reusable; only the semantic layer needs adapting.